In [2]:
!pip install tensorflow==2.15.0 pandas numpy scikit-learn tensorboard matplotlib streamlit

  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [4]:
df=pd.read_csv("Churn_Modelling.csv")

In [5]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
df=df.drop(['RowNumber',	'CustomerId',	'Surname'], axis=1)

In [7]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [13]:
#Encode categorical variables
label_encoder=LabelEncoder()
df['Gender']=label_encoder.fit_transform(df['Gender'])

In [14]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [18]:
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder()
geo_encoder=ohe.fit_transform(df[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [19]:
ohe.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [21]:
geo_encoded_df=pd.DataFrame(geo_encoder.toarray(), columns=ohe.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [ ]:
#Combine One hot encoded part with original data

In [23]:
df=pd.concat([df.drop('Geography', axis=1),geo_encoded_df],axis=1)

In [24]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [28]:
#Save the scaler and encoder
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(ohe, file)

In [29]:
X=df.drop('Exited', axis=1)
y=df['Exited']

#Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [30]:
#Scaling the features
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [31]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

## ANN Implmention

In [40]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [35]:
import datetime

In [38]:
(X_train.shape[1],)

(12,)

In [41]:
#Building the ANN model
model=Sequential([
    Dense(64,activation='relu', input_shape=(X_train.shape[1],)), # HL1 with 64 neurons and ReLU activation
    Dense(32,activation='relu'), # HL2 with 32 neurons and ReLU activation
    Dense(1, activation='sigmoid') # Output layer with 1 neuron and sigmoid activation for binary classification
])

In [42]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [43]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()

In [44]:
#To do bacckpropagation and update weights, we need to compile the model with an optimizer and a loss function
model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])

In [71]:
## Set up TensorBoard callback, this will save logs for visualization in TensorBoard
# What is happening here is that we are creating a log directory with a timestamp to store the logs for this training session. The TensorBoard callback will write logs to this directory during training, which can then be visualized using TensorBoard.
# What is call log is a directory where TensorBoard will save logs related to the training process, such as loss and accuracy metrics, histograms of weights, and other relevant information. The log_dir variable is created by concatenating a base directory ('logs/fit') with a timestamp generated using datetime.datetime.now().strftime('%Y%m%d-%H%M%S'). This ensures that each training session's logs are stored in a unique directory based on the date and time of the training.

from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir='logs/fit/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
tensorflow_callback=TensorBoard(log_dir=log_dir, histogram_freq=1)

In [72]:
# Setting up EarlyStopping callback to monitor validation loss and stop training if it doesn't improve for 5 consecutive epochs. This helps prevent overfitting and saves time by stopping training when the model has stopped improving on the validation set.
early_stopping_callback=EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [73]:
#Training the model with the training data, using a batch size of 32 and training for a maximum of 100 epochs. The validation data is set to the test set, and we are using the early stopping and TensorBoard callbacks to monitor the training process and prevent overfitting.
#Batch size is the number of samples that will be propagated through the network at once. After each batch, the model's weights are updated based on the loss calculated from that batch. This allows for more efficient training and can help with convergence.
history=model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=100,
    callbacks=[early_stopping_callback, tensorflow_callback]
)

Epoch 1/100
250/250 [==============================] - 2s 8ms/step - loss: 0.3060 - accuracy: 0.8700 - val_loss: 0.3774 - val_accuracy: 0.8520
Epoch 2/100
250/250 [==============================] - 2s 7ms/step - loss: 0.3046 - accuracy: 0.8724 - val_loss: 0.3613 - val_accuracy: 0.8550
Epoch 3/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3062 - accuracy: 0.8698 - val_loss: 0.3610 - val_accuracy: 0.8550
Epoch 4/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3015 - accuracy: 0.8699 - val_loss: 0.3511 - val_accuracy: 0.8555
Epoch 5/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3024 - accuracy: 0.8696 - val_loss: 0.3674 - val_accuracy: 0.8580
Epoch 6/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3001 - accuracy: 0.8744 - val_loss: 0.3773 - val_accuracy: 0.8545
Epoch 7/100
250/250 [==============================] - 1s 5ms/step - loss: 0.2935 - accuracy: 0.8751 - val_loss: 0.3645 - val_accuracy: 0.8620

In [74]:
#Save the trained model
model.save('churn_model.h5')

c:\Users\jaisw\Desktop\Projects\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [75]:
#Load TensorBoard Extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [76]:
%tensorboard --logdir logs/fit20260510-212529

Reusing TensorBoard on port 6008 (pid 16352), started 0:04:32 ago. (Use '!kill 16352' to kill it.)

In [77]:
#Load the picle file and trained model
import pickle


model=load_model('churn_model.h5')

#Load the scaler and encoders
with open('scaler.pkl', 'rb') as file:
    scaler=pickle.load(file)
with open('encoders.pkl', 'rb') as file:
    encoders=pickle.load(file)
with open('ohe.pkl', 'rb') as file:
    ohe=pickle.load(file)

NameError: name 'load_model' is not defined

In [ ]:
from tensorflow.keras.models import load_model

model = load_model('churn_model.h5')

# Load the scaler and encoders with correct file names
with open('scaler.pkl', 'rb') as file:
	scaler = pickle.load(file)
with open('label_encoder_gender.pkl', 'rb') as file:
	label_encoder = pickle.load(file)
with open('onehot_encoder_geo.pkl', 'rb') as file:
	ohe = pickle.load(file)